# Foundry Alert Analysis — Fine-Tune Qwen2.5-3B

**Before running:** `Runtime → Change runtime type → T4 GPU` (free)

This notebook fine-tunes Qwen2.5-3B-Instruct on your foundry alert data so it produces
accurate, domain-specific root cause analysis and recommendations.

| Step | What happens |
|---|---|
| 1 | Install Unsloth + dependencies |
| 2 | Load base model with 4-bit quantisation (QLoRA) |
| 3 | Upload & format your `training_data.jsonl` |
| 4 | Train ~200 steps (~30 min on T4) |
| 5 | Test the fine-tuned model |
| 6 | Export to GGUF and download for Ollama |

## Cell 1 — Install dependencies
Run this cell first. **Restart the runtime when prompted**, then continue from Cell 2.

In [ ]:
%%bash
pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
pip install --no-deps trl peft accelerate bitsandbytes xformers -q
echo "✓ Installation complete — restart runtime now"

## Cell 2 — Load model (after runtime restart)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,   # auto-detect (float16 on T4)
    load_in_4bit   = True,   # QLoRA — fits in 6GB VRAM
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,     # LoRA rank — 16 is a good balance
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = 42,
)

print(f"✓ Model loaded — trainable params: {model.num_parameters(only_trainable=True):,}")

## Cell 3 — Upload training data
Upload the `training_data.jsonl` file generated by:
```
python -m watchdog.tools.generate_training_data --limit 500
```

In [ ]:
import json
from google.colab import files
from datasets import Dataset

SYSTEM_PROMPT = """You are an expert in foundry sand preparation monitoring. You receive alert data
from a Sand Index (SI) monitoring system and produce a concise diagnosis.

Domain knowledge:
- Sand properties: active_clay, compactibility, GCS, GFN/AFS, moisture,
  permeability, LOI, volatile_matter, inert_fines, shear_strength, split_strength.
- Additives: bentonite (raises active clay), coal dust/LCA (raises LOI/volatile matter),
  fresh silica sand (controls GFN), water (affects moisture/compactibility).
- Drift = sustained multi-shift trend. Variance = batch-to-batch instability.
- Deviation = outside LCL/UCL control limits.

Respond ONLY with a JSON object:
{\"root_cause\": \"1-2 sentences naming specific parameters and direction\",
 \"recommendation\": \"1-2 sentences — specific, actionable operator steps\"}"""

# Upload the file
print("Select your training_data.jsonl file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Load examples
rows = []
with open(filename, encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

print(f"✓ Loaded {len(rows)} training examples")
print(f"\nSample input:\n{rows[0]['input'][:300]}")
print(f"\nSample output:\n{rows[0]['output'][:200]}")

## Cell 4 — Format dataset with chat template

In [ ]:
def format_example(row):
    """Apply Qwen2.5 chat template to each training example."""
    messages = [
        {"role": "system",    "content": row["instruction"]},
        {"role": "user",      "content": row["input"]},
        {"role": "assistant", "content": row["output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

formatted = [{"text": format_example(r)} for r in rows]
dataset   = Dataset.from_list(formatted)

print(f"✓ Dataset ready — {len(dataset)} examples")
print("\nFormatted sample (first 600 chars):")
print(dataset[0]["text"][:600])

## Cell 5 — Train

| Setting | Value | Note |
|---|---|---|
| `max_steps` | 200 | ~30 min on T4. Set to 500 for better quality (~75 min) |
| `per_device_train_batch_size` | 2 | Fits in 15GB T4 VRAM |
| `learning_rate` | 2e-4 | Standard for LoRA fine-tuning |

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps     = 10,
        max_steps        = 200,         # ← increase to 500 for best quality
        learning_rate    = 2e-4,
        fp16             = True,
        logging_steps    = 10,
        optim            = "adamw_8bit",
        weight_decay     = 0.01,
        lr_scheduler_type = "linear",
        seed             = 42,
        output_dir       = "./output",
    ),
)

# Show GPU memory before training
import torch
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
used    = torch.cuda.memory_allocated(0) / 1e9
print(f"GPU: {gpu_mem:.1f}GB total, {used:.1f}GB used before training")

trainer_stats = trainer.train()

print(f"\n✓ Training complete!")
print(f"  Final loss : {trainer_stats.training_loss:.4f}")
print(f"  Total steps: {trainer_stats.global_step}")

## Cell 6 — Test the fine-tuned model

In [ ]:
import json

FastLanguageModel.for_inference(model)

def ask(alert_text: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": alert_text},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=250,
        temperature=0.3,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

# Test 1 — active clay deviation
test1 = """Alert level: WARNING  |  SI score: 72.4/100
Component: JD RA HSG RH

Non-stable parameters:
  Active Clay:  CRITICAL  drift=STRONG DRIFT  value=7.8  Δ=-4.2%  (Deviated LOW -0.65 (LCL=8.00))
  Moisture:     WATCH     var=ELEVATED        value=3.6  Δ=+1.8%
  GCS:          WATCH     drift=SLIGHT DRIFT  value=2090 Δ=-0.5%"""

print("=" * 60)
print("TEST 1 — Active clay deviation")
print("=" * 60)
out1 = ask(test1)
print(out1)
try:
    parsed = json.loads(out1.strip())
    print(f"\n✓ Valid JSON output")
    print(f"  root_cause     : {parsed['root_cause']}")
    print(f"  recommendation : {parsed['recommendation']}")
except:
    print("⚠ Output is not valid JSON — model may need more training steps")

# Test 2 — oscillation
test2 = """Alert level: ALERT  |  SI score: 58.1/100

Non-stable parameters:
  Compactibility:  ALERT    osc=OSCILLATING   value=39.2  Δ=-2.1%
  Moisture:        ELEVATED var=HIGH VAR      value=3.1   Δ=-3.4%
  Permeability:    WATCH    drift=SLIGHT DRIFT value=88   Δ=-2.0%"""

print("\n" + "=" * 60)
print("TEST 2 — Oscillating compactibility")
print("=" * 60)
print(ask(test2))

## Cell 7 — Export to GGUF for Ollama

This exports the model in a format that Ollama can run locally on your server.
The file will be ~1.8GB.

In [ ]:
print("Exporting to GGUF Q4_K_M (best quality/size tradeoff)...")
print("This takes about 5-10 minutes...")

model.save_pretrained_gguf(
    "foundry-alert-3b",
    tokenizer,
    quantization_method = "q4_k_m",
)

import os
gguf_files = [f for f in os.listdir("foundry-alert-3b") if f.endswith(".gguf")]
for f in gguf_files:
    size = os.path.getsize(f"foundry-alert-3b/{f}") / 1e9
    print(f"  {f}  ({size:.2f} GB)")

print("\n✓ Export complete!")

## Cell 8 — Download the model

In [ ]:
import os
from google.colab import files

gguf_files = [f for f in os.listdir("foundry-alert-3b") if f.endswith(".gguf")]
gguf_path  = f"foundry-alert-3b/{gguf_files[0]}"

print(f"Downloading: {gguf_path}")
print("(This may take a few minutes depending on your internet speed)")
files.download(gguf_path)

print("""
✓ Download complete!

Next steps on your server:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Install Ollama:    https://ollama.com

2. Create Modelfile (save as 'Modelfile' in same folder as .gguf):

   FROM ./foundry-alert-3b-unsloth.Q4_K_M.gguf
   PARAMETER temperature 0.3
   PARAMETER num_ctx 2048

3. Register model:
   ollama create foundry-alert -f Modelfile

4. Test it:
   ollama run foundry-alert

5. Update watchdog_config.json:
   "llm_analysis": {
     "enabled": true,
     "provider": "ollama",
     "ollama_model": "foundry-alert",
     "ollama_url": "http://localhost:11434"
   }
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

## Cell 9 — (Optional) Save adapter to Google Drive instead

If you want to re-train later without re-downloading the base model,
save the LoRA adapter weights to Google Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_PATH = "/content/drive/MyDrive/foundry-alert-adapter"

model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"✓ LoRA adapter saved to Google Drive: {SAVE_PATH}")
print("  To reload later:")
print(f"  model, tokenizer = FastLanguageModel.from_pretrained('{SAVE_PATH}')")